In [0]:
#  con este comando, se pueden validar varias tablas a la vez. Solo hay que indicar el pais y la marca
brand_country = 'MEX'
brand_code = 'SKI'

#### dim_customer
query = f"""
select 
  max(date(created_dt)) max_date,
  count(source_customer_id) conteo
from prod_latam_catalog.crm_reporting.dim_customer
where source_name = 'JEBBIT'
  and brand_country = '{brand_country}'  
  and brand_code = '{brand_code}'
"""
result_df = spark.sql(query)
display(result_df)

#### fact_gdm_beauty_profile
query = f"""
select 
  max(date(sys_created_dt)) max_date,
  count(distinct source_customer_id) conteo
from prod_latam_catalog.crm_reporting.fact_gdm_beauty_profile
where source_name = 'JEBBIT'
  AND brand_country = '{brand_country}'  
  AND brand_code = '{brand_code}'
"""
result_df = spark.sql(query)
display(result_df)

#### dim_gdm_brand_profile
query = f"""
select 
  max(date(a.sys_created_dt)) max_date,
  count(distinct a.brand_mdm_id) conteo  
from prod_latam_catalog.crm_reporting.dim_gdm_brand_profile a
left join prod_latam_catalog.crm_reporting.etl_stg_customer_bridge c
  on a.brand_code = c.brand_code
  and a.brand_country = c.brand_country
  and a.brand_mdm_id = c.brand_mdm_id
where c.source_name = 'JEBBIT'
  AND a.brand_country = '{brand_country}'  
  AND a.brand_code = '{brand_code}'
"""
result_df = spark.sql(query)
display(result_df)

#### dim_customer_bridge
query = f"""
select 
  max(date(a.sys_created_dt)) max_date,
  count(distinct a.brand_customer_id) conteo
from prod_latam_catalog.crm_reporting.fact_beauty_attributes a
left join prod_latam_catalog.crm_reporting.etl_stg_customer_bridge c
  on a.brand_code = c.brand_code
  and a.brand_country = c.brand_country
  and a.brand_customer_id = c.brand_customer_id
where c.source_name = 'JEBBIT'
  and a.brand_country = '{brand_country}'  
  and a.brand_code = '{brand_code}'
"""
result_df = spark.sql(query)
display(result_df)



max_date,conteo
2025-03-13,8680


max_date,conteo
2025-03-13,8825


max_date,conteo
2025-03-13,5811


max_date,conteo
2025-03-13,7451


In [0]:

# AQUI EN ADELANTE valida en la raw por campañas
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType
from pyspark.sql.types import StringType

# Function that identifies and explodes array and struct fields
def explode_arrays(df):
    array_columns = []
    
    # search and select the array columns
    for field in df.schema.fields:
        if isinstance(field.dataType, ArrayType):
            array_columns.append(field.name)

    # loop the array columns
    for col_name in array_columns:
        # Explode each array column
        df = df.withColumn(f"exploded_{col_name}", F.explode_outer(col_name))
        
        # Verify whether the data type of the array is a struct
        element_type = df.schema[f"exploded_{col_name}"].dataType
        if isinstance(element_type, StructType):
            # If it does, extract the struct subfields, create new fields and delete the original one
            for subfield in element_type.fieldNames():
                df = df.withColumn(f"{col_name}_{subfield}", F.col(f"exploded_{col_name}.{subfield}"))
                df = df.drop(col_name)
        else:
            # if the data type is not StructType (ie, StringType o DoubleType), replace the original column with the exploded one
            df = df.withColumn(col_name, F.col(f"exploded_{col_name}"))
        
        # delete the temporal exploded field
        df = df.drop(f"exploded_{col_name}")
    
    return df

# IMPORTAMOS LA RAW
base = spark.table("crm_base.fact_gdm_beauty_profile_raw")
base2 = base.select('brand_country','brand_code','source_customer_id','source_name',"contact_preferences.email",'created_dt')

# explode
explode_df = explode_arrays(base2) # explode first array levels

#
explode_df = explode_df.select('brand_country','brand_code','source_customer_id','source_name','created_dt',"email_campaign_name","email_email").\
  withColumn("created_dt",F.to_date('created_dt')).\
  filter(F.col("source_name").like('%JEBBIT%')).\
  orderBy('created_dt').distinct()

explode_df.createOrReplaceTempView("explode_df_vw")
# display(explode_df)



## CONTEOS POR CAMPAÑA
summary = spark.sql(
    """
select 
  -- source_name,
  brand_country, 
  brand_code,
  email_campaign_name as campaign_name,
  to_date(max(created_dt)) last_entry,
  count(distinct source_customer_id) id_counts
from explode_df_vw
where brand_country = 'MEX' 
  and brand_code = 'SKI'
group by all
order by last_entry desc
    """)

#print(cruce.count())
display(summary)

brand_country,brand_code,campaign_name,last_entry,id_counts
MEX,SKI,SKIN_MEX_SKIN-MEX-Advanced-Regimen-Finder_2023,2025-03-18,5378
MEX,SKI,SKI_MEX_vitamin-C-quiz_2024,2025-03-15,6
MEX,SKI,null,2024-09-16,2
MEX,SKI,SKIN_MX_Find-Your-Perfect-Routine_2022,2024-04-29,3659
MEX,SKI,SKIN_mex_SKIN-mex-Advanced-Regimen-Finder_2023,2023-06-16,1
